# `DocLayNet` FPN-ResNet18 Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` code to recreate a
**Feature Pyramid Network (FPN)** with a **ResNet18** encoder from scratch (~**13M** parameters)!

- FPN paper: https://arxiv.org/abs/1612.03144
- Panoptic FPN (segmentation branch): https://arxiv.org/abs/1901.02446
- ResNet paper: https://arxiv.org/abs/1512.03385

The architecture depends on 3 components:
1. A `ResNet18Encoder` backbone producing the bottom-up features `C2..C5`.
2. A **top-down pathway** with `1x1` lateral connections building the pyramid
   `P2..P5` (all with 256 channels).
3. A **segmentation branch** (Panoptic FPN style): each pyramid level is refined
   to 128 channels at 1/4 scale, the levels are **summed**, and a `1x1` classifier
   + 4x upsampling produces the full-resolution logits.


## 1. The `ResNet18` encoder backbone (from scratch)

We first rebuild the **ResNet18** feature extractor from the original paper
(https://arxiv.org/abs/1512.03385), exactly as in `src/models/backbones.py`.
It is composed of:
- A **stem**: `7x7/2` convolution followed by `3x3/2` max pooling.
- Four residual stages (`layer1..layer4`), each with two `BasicBlock`s
  (two `3x3` convolutions + identity/projection skip connection).

For an input `(B, 3, 512, 512)` the encoder returns 5 multi-scale feature maps:

```python
x  -> stem_conv          -> f1 (B,  64, 256, 256)   # 1/2
f1 -> maxpool + layer1   -> f2 (B,  64, 128, 128)   # 1/4
f2 -> layer2             -> f3 (B, 128,  64,  64)   # 1/8
f3 -> layer3             -> f4 (B, 256,  32,  32)   # 1/16
f4 -> layer4             -> f5 (B, 512,  16,  16)   # 1/32
```

We start with a shared `ConvBNReLU` helper block used across all our benchmark architectures.


In [ ]:
# Let's import all necessary modules for this architecture
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBNReLU(nn.Module):
    '''
    Standard Conv2d -> BatchNorm2d -> ReLU block used across all architectures.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        kernel_size (int): convolution kernel size.
        stride (int): convolution stride.
        padding (int): convolution padding.
        dilation (int): convolution dilation.
        groups (int): convolution groups (used for depthwise convolutions).
        relu6 (bool): if True, uses ReLU6 (MobileNetV2 convention) instead of ReLU.
    '''

    def __init__(self, m: int, n: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, dilation: int = 1, groups: int = 1,
                 relu6: bool = False) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=kernel_size,
                      stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU6() if relu6 else nn.ReLU()
        )

    def forward(self, x) -> torch.Tensor:
        return self.block(x)


In [ ]:
class ResNetBasicBlock(nn.Module):
    '''
    Class that defines the BasicBlock of the ResNet18 architecture
    (two 3x3 convolutions with an identity or projected skip connection).
    Reference: https://arxiv.org/abs/1512.03385

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        stride (int): stride of the first convolution (2 halves the resolution).
    '''

    def __init__(self, m: int, n: int, stride: int = 1) -> None:
        super().__init__()

        # First 3x3 convolution (possibly downsampling)
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(3, 3),
                      stride=(stride, stride), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )

        # Second 3x3 convolution (no activation before the residual add)
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(3, 3),
                      stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True)
        )

        # Projection skip connection when shape changes, identity otherwise
        if stride != 1 or m != n:
            self.skip_conn = nn.Sequential(
                nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(1, 1),
                          stride=(stride, stride), padding=(0, 0), bias=False),
                nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                               affine=True, track_running_stats=True)
            )
        else:
            self.skip_conn = nn.Identity()

        self.relu = nn.ReLU()

    def forward(self, x) -> torch.Tensor:
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = out + self.skip_conn(x)
        return self.relu(out)


In [ ]:
class ResNet18Encoder(nn.Module):
    '''
    Class that defines the full ResNet18 feature-extractor backbone from scratch
    (no fully connected head), returning multi-scale feature maps.
    Reference: https://arxiv.org/abs/1512.03385

    Feature maps returned for an input of shape (B, Cin, H, W):
        f1: (B,  64, H/2,  W/2)   -> after stem conv (before max pooling)
        f2: (B,  64, H/4,  W/4)   -> after layer1
        f3: (B, 128, H/8,  W/8)   -> after layer2
        f4: (B, 256, H/16, W/16)  -> after layer3
        f5: (B, 512, H/32, W/32)  -> after layer4

    Args:
        Cin (int): number of input channels (3 for RGB document images).
    '''

    # Output channels at each stage, useful for building decoders
    out_channels: List[int] = [64, 64, 128, 256, 512]

    def __init__(self, Cin: int = 3) -> None:
        super().__init__()

        # Stem: 7x7/2 convolution followed by 3x3/2 max pooling
        self.stem_conv = nn.Sequential(
            nn.Conv2d(in_channels=Cin, out_channels=64, kernel_size=(7, 7),
                      stride=(2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(num_features=64, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )
        self.max_pool = nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

        # Four residual stages, two BasicBlocks each (ResNet18 configuration)
        self.layer1 = nn.Sequential(
            ResNetBasicBlock(m=64, n=64, stride=1),
            ResNetBasicBlock(m=64, n=64, stride=1)
        )
        self.layer2 = nn.Sequential(
            ResNetBasicBlock(m=64, n=128, stride=2),
            ResNetBasicBlock(m=128, n=128, stride=1)
        )
        self.layer3 = nn.Sequential(
            ResNetBasicBlock(m=128, n=256, stride=2),
            ResNetBasicBlock(m=256, n=256, stride=1)
        )
        self.layer4 = nn.Sequential(
            ResNetBasicBlock(m=256, n=512, stride=2),
            ResNetBasicBlock(m=512, n=512, stride=1)
        )

    def forward(self, x) -> List[torch.Tensor]:
        f1 = self.stem_conv(x)          # (B, 64, H/2, W/2)
        f2 = self.layer1(self.max_pool(f1))  # (B, 64, H/4, W/4)
        f3 = self.layer2(f2)            # (B, 128, H/8, W/8)
        f4 = self.layer3(f3)            # (B, 256, H/16, W/16)
        f5 = self.layer4(f4)            # (B, 512, H/32, W/32)
        return [f1, f2, f3, f4, f5]


### 1.1 Summary info of `ResNet18Encoder`

In [ ]:
from torchinfo import summary
# Let's inspect the ResNet18 encoder backbone
test_model = ResNet18Encoder(Cin=3)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## 2. The top-down pathway and lateral connections

Each coarser pyramid level is upsampled 2x (nearest) and **added** to a `1x1`
projection of the corresponding encoder feature:

```python
f5 (B, 512, 16, 16) -> 1x1 conv                    -> P5 (B, 256, 16, 16)
P5 up 2x + lateral(f4, 256 -> 256)                 -> P4 (B, 256, 32, 32)
P4 up 2x + lateral(f3, 128 -> 256)                 -> P3 (B, 256, 64, 64)
P3 up 2x + lateral(f2,  64 -> 256)                 -> P2 (B, 256, 128, 128)
```


In [ ]:
class FPNLateralBlock(nn.Module):
    '''
    Class that defines the FPN top-down pathway block: a 1x1 lateral
    convolution on the encoder feature plus the 2x upsampled coarser
    pyramid feature.

    Args:
        m (int): number of input channels of the encoder skip feature.
        pyramid_channels (int): number of channels of every pyramid level.
    '''

    def __init__(self, m: int, pyramid_channels: int = 256) -> None:
        super().__init__()
        self.lateral_conv = nn.Conv2d(in_channels=m, out_channels=pyramid_channels,
                                      kernel_size=(1, 1), stride=(1, 1), padding=(0, 0))

    def forward(self, x, skip) -> torch.Tensor:
        # Upsample the coarser pyramid feature and add the lateral projection
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return x + self.lateral_conv(skip)


## 3. The segmentation branch

Following Panoptic FPN, each level is processed by repeated
(`3x3` conv -> GroupNorm -> ReLU -> 2x upsample) blocks until it reaches **1/4**
of the input resolution (P5 needs 3 upsamples, P4 needs 2, P3 needs 1, P2 none),
then all four maps are summed:

```python
P5 (B, 256, 16, 16)   -> seg_block (3 ups) -> (B, 128, 128, 128)
P4 (B, 256, 32, 32)   -> seg_block (2 ups) -> (B, 128, 128, 128)
P3 (B, 256, 64, 64)   -> seg_block (1 ups) -> (B, 128, 128, 128)
P2 (B, 256, 128, 128) -> seg_block (0 ups) -> (B, 128, 128, 128)
sum -> dropout -> 1x1 conv -> N logits -> 4x upsample -> (B, N, 512, 512)
```


In [ ]:
class FPNSegmentationBlock(nn.Module):
    '''
    Class that defines the Panoptic-FPN segmentation branch block: each
    pyramid level is processed by (3x3 conv -> GroupNorm -> ReLU -> 2x upsample)
    repeated until it reaches 1/4 of the input resolution.

    Args:
        m (int): number of input channels (pyramid channels).
        n (int): number of output channels (segmentation channels).
        num_upsamples (int): how many 2x upsampling steps to reach 1/4 scale.
    '''

    def __init__(self, m: int, n: int, num_upsamples: int = 0) -> None:
        super().__init__()
        blocks = []
        for i in range(max(1, num_upsamples)):
            in_ch = m if i == 0 else n
            blocks.append(nn.Sequential(
                nn.Conv2d(in_channels=in_ch, out_channels=n, kernel_size=(3, 3),
                          stride=(1, 1), padding=(1, 1), bias=False),
                nn.GroupNorm(num_groups=32, num_channels=n),
                nn.ReLU(),
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
                if i < num_upsamples else nn.Identity()
            ))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x) -> torch.Tensor:
        return self.blocks(x)


## 4. Final step, let's create the entire network


In [ ]:
class FPNResNet18Model(nn.Module):
    '''
    Class that defines the full FPN architecture with a ResNet18 encoder.
    The encoder features C2..C5 are merged in a top-down pyramid P2..P5
    (256 channels), each level is refined to 128 segmentation channels at
    1/4 scale, summed, and upsampled to full resolution.

    Args:
        Cin (int): number of input channels for the encoder.
        N (int): number of output channels (1 binary / num_classes semantic).
        pyramid_channels (int): channels of the pyramid levels (default 256).
        segmentation_channels (int): channels of the segmentation branch (default 128).
    '''

    def __init__(self, Cin: int = 3, N: int = 1,
                 pyramid_channels: int = 256,
                 segmentation_channels: int = 128) -> None:
        super().__init__()
        self.encoder = ResNet18Encoder(Cin=Cin)
        c1, c2, c3, c4, c5 = self.encoder.out_channels

        # Top of the pyramid: 1x1 projection of the deepest feature (C5 -> P5)
        self.p5_conv = nn.Conv2d(in_channels=c5, out_channels=pyramid_channels,
                                 kernel_size=(1, 1), stride=(1, 1), padding=(0, 0))

        # Top-down pathway with lateral connections (C4 -> P4, C3 -> P3, C2 -> P2)
        self.p4_block = FPNLateralBlock(m=c4, pyramid_channels=pyramid_channels)
        self.p3_block = FPNLateralBlock(m=c3, pyramid_channels=pyramid_channels)
        self.p2_block = FPNLateralBlock(m=c2, pyramid_channels=pyramid_channels)

        # Segmentation branch: refine every level to 1/4 of the input resolution
        self.seg_block_5 = FPNSegmentationBlock(m=pyramid_channels, n=segmentation_channels, num_upsamples=3)
        self.seg_block_4 = FPNSegmentationBlock(m=pyramid_channels, n=segmentation_channels, num_upsamples=2)
        self.seg_block_3 = FPNSegmentationBlock(m=pyramid_channels, n=segmentation_channels, num_upsamples=1)
        self.seg_block_2 = FPNSegmentationBlock(m=pyramid_channels, n=segmentation_channels, num_upsamples=0)

        # Final classifier: dropout + 1x1 conv, then 4x upsample to full resolution
        self.dropout = nn.Dropout2d(p=0.2)
        self.segmentation_head = nn.Conv2d(in_channels=segmentation_channels,
                                           out_channels=N, kernel_size=(1, 1),
                                           stride=(1, 1), padding=(0, 0))

    def forward(self, x) -> torch.Tensor:
        _, f2, f3, f4, f5 = self.encoder(x)

        # Build the pyramid top-down
        p5 = self.p5_conv(f5)
        p4 = self.p4_block(p5, f4)
        p3 = self.p3_block(p4, f3)
        p2 = self.p2_block(p3, f2)

        # Merge all levels at 1/4 resolution by summation
        s = self.seg_block_5(p5) + self.seg_block_4(p4) \
            + self.seg_block_3(p3) + self.seg_block_2(p2)

        s = self.dropout(s)
        s = self.segmentation_head(s)

        # Upsample from 1/4 back to full input resolution
        return F.interpolate(s, scale_factor=4, mode="bilinear", align_corners=True)


### 4.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [ ]:
from torchinfo import summary
# Full FPN-ResNet18 model at 1024x1024
test_model = FPNResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


### 4.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [ ]:
from torchinfo import summary
# Full FPN-ResNet18 model at 512x512
test_model = FPNResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## Integration with `src/models` and the training framework

The exact same classes above live in `src/models`, and the `fpn-resnet18` model can be
built through the shared model factory. This is what the training scripts use when you pass
`--arch fpn-resnet18`:

```bash
python scripts/train_binary_text.py --arch fpn-resnet18
python scripts/train_semantic_layout.py --arch fpn-resnet18
```

Let's double check the factory produces the same model, and run the IEEE efficiency
benchmark (parameters, FLOPs, inference latency/FPS, and peak memory - GPU if available,
otherwise CPU RSS) with `src.utils.benchmark`.


In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root (src.*)
sys.path.insert(0, str(Path().cwd().parent))

from src.models import build_model

factory_model = build_model("fpn-resnet18", Cin=3, N=1)
total_params = sum(p.numel() for p in factory_model.parameters())
print(f"FPN-ResNet18 total parameters: {total_params:,} ({total_params/1e6:.2f}M)")


In [ ]:
from src.utils import benchmark_model, print_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
report = benchmark_model(
    model=factory_model,
    input_size=(1, 3, 512, 512),
    device=device,
    warmup=5,
    iterations=20,
    arch_name="fpn-resnet18",
)
print_benchmark(report)
